# 🚀 Giai đoạn 4: Đánh giá Benchmark Pretrained Transformer (ViSoBERT)
Đánh giá Zero-shot checkpoint Pretrained ViSoBERT cho phân loại cảm xúc tiếng Việt trên dữ liệu ITviec


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

model_name = "5CD-AI/Vietnamese-Sentiment-visobert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
sentiment_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

sample_text = "Công ty môi trường làm việc chuyên nghiệp, đồng nghiệp thân thiện, sếp tâm lý."
print(sentiment_pipe(sample_text))


/root/NLP-Sentiment-Analysis-ITviec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'label': 'POS', 'score': 0.9966692328453064}]


> **Kết quả:** benchmark zero-shot (không fine-tune) checkpoint `5CD-AI/Vietnamese-Sentiment-visobert` trên GPU (Runpod). Số liệu: Accuracy 0.6536, Macro F1 0.4036 (xem so sánh chi tiết với Stacking Ensemble ở `reports/modeling_hyperparameter_tuning.md`).

## 4.1 Benchmark Zero-shot trên đúng tập Final Test đã khóa của notebook 03
Dùng lại `test_indices` từ `models/train_test_features.joblib` để lấy đúng các dòng final test mà notebook 03 dùng, đảm bảo so sánh ViSoBERT với các mô hình ML là công bằng (cùng tập test, cùng nhãn thật). Văn bản đưa vào ViSoBERT dùng cột `clean_basic_text` (giữ nguyên cấu trúc câu tự nhiên) thay vì `clean_advance_text` (đã tách từ ghép, chỉ tối ưu cho ML cổ điển) — theo đúng thiết kế 2 tầng tiền xử lý trong README.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

from src.features import load_feature_split

artifact = load_feature_split(PROJECT_ROOT / 'models' / 'train_test_features.joblib')
test_indices = artifact['test_indices']
y_test = artifact['y_test']

reviews = pd.read_excel(PROJECT_ROOT / 'data' / 'processed' / 'reviews_cleaned.xlsx')
test_texts = reviews.loc[test_indices, 'clean_basic_text'].fillna('').astype(str)
print(f'So mau final test: {len(test_texts)}')

So mau final test: 1683


In [3]:
# Nhan tho ma pipeline tra ve tuy theo checkpoint (vd 'POS'/'NEU'/'NEG' hoac
# 'LABEL_0'/'LABEL_1'/'LABEL_2'). Chay cell nay tren GPU truoc de biet chinh xac
# nhan tho la gi, roi chinh sua RAW_LABEL_TO_SENTIMENT cho khop neu can.
RAW_LABEL_TO_SENTIMENT = {
    'POS': 'Positive', 'NEU': 'Neutral', 'NEG': 'Negative',
    'POSITIVE': 'Positive', 'NEUTRAL': 'Neutral', 'NEGATIVE': 'Negative',
    'LABEL_0': 'Negative', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positive',
}

batch_size = 32
raw_predictions = []
for start in range(0, len(test_texts), batch_size):
    batch = test_texts.iloc[start:start + batch_size].tolist()
    outputs = sentiment_pipe(batch, truncation=True, max_length=256)
    raw_predictions.extend(item['label'] for item in outputs)

unseen_labels = set(raw_predictions) - set(RAW_LABEL_TO_SENTIMENT)
if unseen_labels:
    raise ValueError(
        f'Nhan {unseen_labels} chua co trong RAW_LABEL_TO_SENTIMENT, '
        'hay bo sung anh xa roi chay lai.'
    )

y_pred_visobert = [RAW_LABEL_TO_SENTIMENT[label] for label in raw_predictions]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


## 4.2 Đánh giá & so sánh với các mô hình ML (notebook 03)

In [4]:
visobert_f1_macro = f1_score(y_test, y_pred_visobert, average='macro')
visobert_accuracy = accuracy_score(y_test, y_pred_visobert)

print(f'ViSoBERT (zero-shot) - Accuracy: {visobert_accuracy:.4f}; Macro F1: {visobert_f1_macro:.4f}')
print(classification_report(y_test, y_pred_visobert, digits=4))

ViSoBERT (zero-shot) - Accuracy: 0.6536; Macro F1: 0.4036
              precision    recall  f1-score   support

    Negative     0.1882    0.7544    0.3012       114
     Neutral     0.3902    0.0488    0.0867       328
    Positive     0.8422    0.8042    0.8228      1241

    accuracy                         0.6536      1683
   macro avg     0.4735    0.5358    0.4036      1683
weighted avg     0.7098    0.6536    0.6440      1683



## 4.3 Phân tích hạn chế tiền xử lý & Đề xuất pipeline chuẩn hóa cho Transformer

Qua phân tích lỗi (Error Analysis), nhóm phát hiện một nguyên nhân cốt lõi khiến ViSoBERT zero-shot chưa đạt hiệu năng tối ưu:
- Pipeline `clean_basic_text` ban đầu được thiết kế theo tư duy ML truyền thống: xóa toàn bộ dấu câu (làm mất ranh giới phân tách ý của Self-Attention), ép emoji thành từ ngữ và dịch teencode.
- Để khắc phục và làm tiền đề vững chắc cho việc Fine-tuning ViSoBERT, nhóm đã xây dựng phương thức chuyên biệt `clean_text_for_transformer()` trong `src/preprocessing.py`:
  - **Bảo tồn dấu câu và ngữ pháp (. , !):** Giúp Transformer định vị chính xác cấu trúc câu.
  - **Bảo tồn Emoji tự nhiên:** Tận dụng kho vector biểu cảm có sẵn trong bộ từ vựng BPE của ViSoBERT.
  - **Rút gọn ký tự lặp kéo dài:** Rút gọn về tối đa 2 ký tự (vd: `vuiiiii` -> `vuii`).
  - **Bảo toàn ngôn ngữ tự nhiên:** Giữ nguyên teencode và thuật ngữ IT (không dịch thô làm mất ngữ cảnh).

In [5]:
from src.preprocessing import TextPreprocessor

prep = TextPreprocessor()
sample = "Công ty IT service này vuiiiii quáaaa! Sếp PM tốt, ko bắt OT nhiều 😡."

print("1. Văn bản gốc (Raw review):")
print(sample)
print("\n2. Qua clean_basic_text (tối ưu cho TF-IDF / Machine Learning):")
print(prep.clean_basic_text(sample))
print("\n3. Qua clean_text_for_transformer (tối ưu cho Pretrained Transformer):")
print(prep.clean_text_for_transformer(sample))


1. Văn bản gốc (Raw review):
Công ty IT service này vuiiiii quáaaa! Sếp PM tốt, ko bắt OT nhiều 😡.

2. Qua clean_basic_text (tối ưu cho TF-IDF / Machine Learning):
công ty công nghệ thông tin nó dịch vụ này vuii quáa sếp quản lý dự án tốt không bắt làm thêm giờ nhiều giận dữ tức giận

3. Qua clean_text_for_transformer (tối ưu cho Pretrained Transformer):
Công ty IT service này vuii quáa! Sếp PM tốt, ko bắt OT nhiều 😡.
